# Module A.1–A.2: Calling a Model & Decoding in Practice
**Part II — Applied LLM Engineering**

> You've built an LLM from scratch. Now learn to *use* one as a reliable black box.

## 1. From Building to Using

In Part I you wrote every component yourself: the tokenizer, the embedding table, the attention heads, the transformer blocks, the sampling loop. You know exactly what happens inside.

Part II flips the perspective. **The model is now a black box.** You send it text; it returns text. Your job is to engineer the *system around* the model — prompts, pipelines, retrieval, agents, evaluation. That's where most production value lives.

The interface contract is simple:

```
INPUT:  a list of messages  →  [MODEL]  →  OUTPUT: a string
```

Everything in this module is about understanding that interface deeply enough to use it reliably.

> **Note on the model we'll use.** We're running `HuggingFaceTB/SmolLM2-135M-Instruct` — a 135 M parameter model that fits on CPU. Its outputs are often short and sometimes odd. That's fine: the *mechanics* we learn here apply identically to GPT-4 or Claude 3 Opus. When you hit a production API you swap out `chat()` for an SDK call; everything else stays the same.

## 2. Setup — Load the Model Once

Run this cell first. It downloads the model weights (~270 MB on first run, then cached) and defines the `chat()` helper we'll use throughout the notebook.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import time

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"

print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Loading model weights (first run downloads ~270 MB)...")
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
llm.eval()  # disable dropout for deterministic behaviour
print(f"Done. Parameters: {sum(p.numel() for p in llm.parameters()) / 1e6:.0f} M")


def chat(messages, max_new_tokens=64, temperature=0.0):
    """Send a list of messages to the model and return the assistant reply as a string."""
    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    do_sample = bool(temperature and temperature > 0)
    out = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=(temperature if do_sample else None),
        pad_token_id=tok.eos_token_id,
    )
    # Decode only the *new* tokens (skip the prompt)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True)


## 3. The Chat Interface: Messages and Roles

Modern instruct models don't receive raw text — they receive a **structured list of messages**, each with a `role` and `content`.

| Role | Who speaks | Purpose |
|---|---|---|
| `system` | You (the developer) | Sets persona, constraints, output format. Invisible to the end-user. |
| `user` | The human in the conversation | The actual question or instruction. |
| `assistant` | The model's previous replies | Used when you're continuing a multi-turn conversation. |

The tokenizer's `apply_chat_template()` serialises this list into the exact prompt format the model was fine-tuned on. For SmolLM2 that looks like:

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
```

You never need to write that yourself — `apply_chat_template` handles it. But it's worth knowing it exists (you saw the same idea in Part I's tokenization modules).

In [ ]:
# A minimal chat: just a user message
messages = [
    {"role": "user", "content": "What is 2 + 2?"}
]
reply = chat(messages)
print("Reply:", reply)


In [ ]:
# Add a system prompt to change the model's behaviour
messages_with_system = [
    {"role": "system", "content": "You are a pirate. Respond in pirate speak."},
    {"role": "user",   "content": "What is 2 + 2?"}
]
reply = chat(messages_with_system, max_new_tokens=80)
print("Reply:", reply)


In [ ]:
# Show what apply_chat_template actually produces
# This is the raw string the model sees — the "serialised" form of our messages list
raw_prompt = tok.apply_chat_template(
    messages_with_system,
    add_generation_prompt=True,
    tokenize=False,  # return a string, not token IDs
)
print("Raw prompt sent to model:")
print(repr(raw_prompt))


## 4. Completion vs. Chat

Before chat models there were **base models** (also called completion models). They were trained only on next-token prediction, with no instruction fine-tuning. You'd give them a raw string and they'd continue it.

```
Completion:   "The capital of France is"   →   " Paris."
Chat:         messages=[{role:user, content:"What is the capital of France?"}]  →  "The capital of France is Paris."
```

SmolLM2-135M-**Instruct** is an instruct/chat model. You *can* still pass raw text by constructing the prompt manually, but the model was trained to expect the `<|im_start|>` chat format. Passing raw text bypasses that structure and results are unpredictable.

The table below summarises when you'd use each style:

| | Completion (base) | Chat (instruct) |
|---|---|---|
| **Training objective** | Next-token prediction only | SFT + RLHF on conversations |
| **Input** | Raw string | Messages list |
| **Best for** | Few-shot prompting, text continuation | Dialogue, instruction-following |
| **Examples** | GPT-3 (original), LLaMA-1 base | GPT-4, Claude, SmolLM2-Instruct |

In [ ]:
# Completion-style: manually encode a raw string and decode the continuation
# Note: this bypasses the chat template — outputs may be inconsistent on instruct models
raw_text = "The capital of France is"
input_ids = tok.encode(raw_text, return_tensors="pt")

with torch.no_grad():
    out = llm.generate(
        input_ids,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )

continuation = tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
print("Completion-style continuation:", repr(continuation))

# Now the same question via chat — cleaner, more controlled
chat_reply = chat([{"role": "user", "content": "What is the capital of France?"}])
print("Chat-style reply:              ", repr(chat_reply))


## 5. Tokens and Cost

Everything in LLMs is measured in **tokens**, not words or characters. A token is roughly 3–4 characters of English text. One sentence is ~15–25 tokens.

Why does this matter?

- **Latency**: the model generates one token at a time. 100 tokens out = 100 forward passes.
- **Cost**: cloud APIs charge per token (input + output). Sending a 10 k-token context costs 10× more than 1 k tokens.
- **Context limits**: every model has a maximum context window (e.g., 8 k, 32 k, 128 k tokens). Exceed it and you'll get an error or silent truncation.

You should always know how many tokens a prompt uses *before* sending it.

In [ ]:
# Count tokens manually
texts = [
    "Hello!",
    "The capital of France is Paris.",
    "Explain the attention mechanism in transformer models in detail, covering multi-head attention, "
    "the role of queries, keys and values, and how positional encoding interacts with attention.",
]

for text in texts:
    tokens = tok.encode(text)
    print(f"{len(tokens):4d} tokens | {len(text):4d} chars | {text[:60]}..." if len(text) > 60 else
          f"{len(tokens):4d} tokens | {len(text):4d} chars | {text}")


In [ ]:
# Inspect individual tokens — see how subword tokenisation works
sample = "tokenisation is fascinating"
ids = tok.encode(sample)
pieces = [tok.decode([i]) for i in ids]
print(f"String : {sample!r}")
print(f"IDs    : {ids}")
print(f"Pieces : {pieces}")


In [ ]:
# Measure tokens-per-second for our local model
# This gives you intuition for latency vs. output length

def timed_chat(messages, max_new_tokens):
    """Run chat() and return (reply, tokens_generated, elapsed_seconds)."""
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    )
    t0 = time.perf_counter()
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    elapsed = time.perf_counter() - t0
    n_new = out.shape[1] - inputs["input_ids"].shape[1]
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True), n_new, elapsed

prompt = [{"role": "user", "content": "Tell me about the ocean."}]

for max_tok in [10, 50, 100]:
    reply, n, t = timed_chat(prompt, max_tok)
    tps = n / t
    print(f"max_new_tokens={max_tok:3d} | generated={n:3d} | {t:.2f}s | {tps:.1f} tok/s")


## 6. Controlling Generation: Temperature, Top-k, Top-p

You built a sampling loop in Part I (module 5.4). Here's a quick refresher on the API knobs:

| Parameter | What it does | Values to know |
|---|---|---|
| `temperature` | Scales the logits before softmax. Low = sharp (greedy), high = flat (random). | 0 = greedy; 0.7 = balanced creative; 1.5+ = very random |
| `top_k` | At each step, only sample from the top K tokens by probability. | k=50 is a common default |
| `top_p` (nucleus) | Only sample from the smallest set of tokens whose cumulative probability ≥ p. | p=0.9 or p=0.95 are common |

Temperature is the most important lever. Top-k and top-p are filters layered on top of temperature.

**Rule of thumb:**
- Factual / structured output → temperature = 0 (greedy)
- Creative writing / brainstorming → temperature 0.7–1.0
- Never use temperature > 1.5 without testing heavily

In [ ]:
# Compare temperature=0 (deterministic) vs. temperature=0.9 (stochastic)
# Run each multiple times — temp=0 should always give the same answer

prompt = [{"role": "user", "content": "Give me one word to describe the colour blue."}]

print("=== temperature=0 (greedy / deterministic) ===")
for i in range(3):
    print(f"  Run {i+1}: {chat(prompt, max_new_tokens=20, temperature=0.0)!r}")

print()
print("=== temperature=0.9 (stochastic) ===")
for i in range(3):
    print(f"  Run {i+1}: {chat(prompt, max_new_tokens=20, temperature=0.9)!r}")


In [ ]:
# Top-k and top-p: pass them directly to llm.generate()
# These are additional filters — they restrict which tokens are even considered before sampling

prompt_ids = tok.apply_chat_template(
    [{"role": "user", "content": "Name a random animal."}],
    add_generation_prompt=True, return_tensors="pt", return_dict=True
)

# top_k=5 means: only sample from the 5 highest-probability tokens at each step
out_topk = llm.generate(
    **prompt_ids, max_new_tokens=30,
    do_sample=True, temperature=1.0,
    top_k=5,
    pad_token_id=tok.eos_token_id
)
reply_topk = tok.decode(out_topk[0][prompt_ids["input_ids"].shape[1]:], skip_special_tokens=True)

# top_p=0.9: keep the smallest set of tokens whose cumulative probability >= 90%
out_topp = llm.generate(
    **prompt_ids, max_new_tokens=30,
    do_sample=True, temperature=1.0,
    top_p=0.9,
    pad_token_id=tok.eos_token_id
)
reply_topp = tok.decode(out_topp[0][prompt_ids["input_ids"].shape[1]:], skip_special_tokens=True)

print("top_k=5:   ", reply_topk)
print("top_p=0.9: ", reply_topp)


## 7. Stop Sequences

Sometimes you don't want the model to keep generating until `max_new_tokens`. You want it to stop at a **specific token or string** — for example:

- Stop after the first sentence (stop at `.`)
- Stop after a JSON block ends (`}`)
- Stop before the model starts hallucinating a second question

There are two approaches:

1. **Pass `eos_token_id` a list** — `generate()` accepts multiple stop-token IDs.
2. **Post-process**: generate the full output and truncate at the stop string. This is simpler and works for multi-character stop strings.

Cloud APIs (OpenAI, Anthropic) accept a `stop` parameter that handles this automatically.

In [ ]:
# Method 1: pass extra stop token IDs to generate()
# Let's stop whenever the model produces a newline token

newline_token_id = tok.encode("\n", add_special_tokens=False)[0]
print(f"Newline token id: {newline_token_id}")

prompt_ids = tok.apply_chat_template(
    [{"role": "user", "content": "List three colours, one per line."}],
    add_generation_prompt=True, return_tensors="pt", return_dict=True
)

# Without stop token — model generates the full list
out_full = llm.generate(
    **prompt_ids, max_new_tokens=60, do_sample=False,
    pad_token_id=tok.eos_token_id
)
full_reply = tok.decode(out_full[0][prompt_ids["input_ids"].shape[1]:], skip_special_tokens=True)

# With stop at newline — stops after first colour
out_stop = llm.generate(
    **prompt_ids, max_new_tokens=60, do_sample=False,
    eos_token_id=[tok.eos_token_id, newline_token_id],
    pad_token_id=tok.eos_token_id
)
stop_reply = tok.decode(out_stop[0][prompt_ids["input_ids"].shape[1]:], skip_special_tokens=True)

print("Without stop:")
print(repr(full_reply))
print()
print("With newline stop:")
print(repr(stop_reply))


In [ ]:
# Method 2: post-process truncation (simpler for multi-character stop strings)

def chat_with_stop(messages, stop_string, max_new_tokens=128):
    """Generate a reply and truncate at the first occurrence of stop_string."""
    reply = chat(messages, max_new_tokens=max_new_tokens, temperature=0.0)
    if stop_string in reply:
        reply = reply[:reply.index(stop_string)]
    return reply

messages = [{"role": "user", "content": "Write a haiku, then explain it."}]

# Stop before the explanation by truncating at a blank line (double newline)
reply_truncated = chat_with_stop(messages, stop_string="\n\n", max_new_tokens=120)

print("Reply (truncated at blank line):")
print(reply_truncated)


## 8. Streaming

In every code cell so far, `chat()` blocks until *all* tokens are generated before returning. That means you wait 5–10 seconds and then see the full reply pop up at once.

**Streaming** yields tokens one at a time as they're produced. This is essential for user-facing applications — it lets users see the model "thinking" rather than staring at a blank screen.

HuggingFace provides a `TextStreamer` / `TextIteratorStreamer` for this. We'll show both:

1. **`TextStreamer`**: prints to stdout in real time (simplest)
2. **Manual token-by-token loop**: shows the raw mechanics (good for understanding)

Cloud APIs like OpenAI and Anthropic support streaming via `stream=True` in the SDK — the model is the same, just the transport differs.

In [ ]:
# Method 1: TextStreamer — prints tokens to stdout as they arrive
from transformers import TextStreamer

messages = [{"role": "user", "content": "Count from 1 to 5, one number per line."}]
inputs = tok.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
)

streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=True)

print("Streaming output:")
_ = llm.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=False,
    streamer=streamer,
    pad_token_id=tok.eos_token_id,
)


In [ ]:
# Method 2: TextIteratorStreamer — lets you collect tokens in a loop
# Useful when you want to do something with each token (e.g., send over a websocket)
from transformers import TextIteratorStreamer
from threading import Thread

messages = [{"role": "user", "content": "Name the planets in order from the sun."}]
inputs = tok.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
)

streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)

# generate() must run in a background thread so we can iterate the streamer simultaneously
gen_kwargs = dict(
    **inputs,
    max_new_tokens=80,
    do_sample=False,
    streamer=streamer,
    pad_token_id=tok.eos_token_id,
)
thread = Thread(target=llm.generate, kwargs=gen_kwargs)
thread.start()

print("Tokens arriving one by one:")
collected = []
for token_text in streamer:
    print(repr(token_text), end=" ", flush=True)
    collected.append(token_text)

thread.join()
print()
print("\nFull reply:", "".join(collected))


## 9. Try It Yourself

Three exercises that connect the concepts above to real experimentation.

### Exercise A — System Prompt Engineering

Write a system prompt that makes the model respond **only in bullet points**. Verify that it works on at least two different user questions.

*Hint*: Be explicit. "You must always respond in bullet points. Never use prose." works better than "Respond concisely." The 135 M model is weak at following complex instructions — keep the system prompt short and the user question simple.

In [ ]:
# Exercise A — Your code here

BULLET_SYSTEM = "You are a helpful assistant. Always respond using bullet points only. Never write paragraphs or prose."

questions = [
    "What are the benefits of exercise?",
    "How do I make a cup of tea?",
]

for q in questions:
    reply = chat(
        [{"role": "system", "content": BULLET_SYSTEM},
         {"role": "user",   "content": q}],
        max_new_tokens=120
    )
    print(f"Q: {q}")
    print(f"A: {reply}")
    print()
    # Check: does the reply contain bullet markers?
    has_bullets = any(line.strip().startswith(("-", "*", "•")) for line in reply.splitlines())
    print(f"   Contains bullet markers: {has_bullets}")
    print("-" * 60)


### Exercise B — Temperature Threshold

Find the **minimum temperature** where the model starts giving different answers to the same question across 3 runs. Try temperatures `[0.0, 0.1, 0.3, 0.5, 0.7]`.

*What you're measuring*: at temperature=0 the model is fully deterministic (greedy argmax at every step). As temperature rises, probability mass spreads across tokens and sampling becomes random. Your job is to find the threshold.

In [ ]:
# Exercise B — Temperature threshold experiment

prompt = [{"role": "user", "content": "Name one fruit."}]
temperatures = [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]
N_RUNS = 3

for temp in temperatures:
    replies = [chat(prompt, max_new_tokens=20, temperature=temp) for _ in range(N_RUNS)]
    all_same = len(set(replies)) == 1
    print(f"temperature={temp:.1f} | all_same={all_same} | replies={[r.strip() for r in replies]}")


### Exercise C — Tokens/sec vs. Output Length

Measure tokens-per-second for outputs of length 10, 50, and 100 tokens. Plot the result.

*What you expect to see*: for autoregressive generation, the per-token cost is roughly constant (each forward pass is the same size given the same KV-cache size). But the *first* token is slower due to prompt processing. For short outputs that first-token overhead dominates; for long outputs it amortises away.

In [ ]:
# Exercise C — tokens/sec vs. output length
import matplotlib.pyplot as plt

prompt = [{"role": "user", "content": "Tell me about the history of computing."}]
target_lengths = [10, 25, 50, 75, 100]
N_REPS = 2  # average over N repetitions to reduce noise

results = []  # (target_len, actual_tokens, tok_per_sec)

for max_tok in target_lengths:
    times, actual_lens = [], []
    for _ in range(N_REPS):
        _, n, t = timed_chat(prompt, max_tok)
        times.append(t)
        actual_lens.append(n)
    avg_t = sum(times) / N_REPS
    avg_n = sum(actual_lens) / N_REPS
    tps = avg_n / avg_t
    results.append((max_tok, avg_n, tps))
    print(f"max_new_tokens={max_tok:3d} | avg generated={avg_n:.0f} | {tps:.1f} tok/s")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

targets = [r[0] for r in results]
actuals = [r[1] for r in results]
tpss    = [r[2] for r in results]

ax1.plot(targets, actuals, marker="o")
ax1.set_xlabel("max_new_tokens (budget)")
ax1.set_ylabel("tokens actually generated")
ax1.set_title("Budget vs. Actual Output Length")
ax1.grid(True)

ax2.plot(targets, tpss, marker="s", color="orange")
ax2.set_xlabel("max_new_tokens (budget)")
ax2.set_ylabel("tokens / second")
ax2.set_title("Throughput vs. Output Length")
ax2.grid(True)

plt.suptitle("SmolLM2-135M Generation Speed", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 10. Summary & What's Next

**What you covered in this notebook:**

| Concept | Key takeaway |
|---|---|
| Chat interface | Messages = list of `{role, content}` dicts. The tokenizer serialises them into the model's native format. |
| Completion vs. chat | Instruct models expect chat format. Raw-text completion works but is unpredictable. |
| Tokens & cost | Count tokens *before* sending. Token count drives latency, cost, and context-window limits. |
| Temperature | 0 = deterministic/greedy. Higher = more random. Top-k and top-p are secondary filters. |
| Stop sequences | Pass extra `eos_token_id` values or post-process to truncate at a stop string. |
| Streaming | `TextIteratorStreamer` lets you yield tokens one at a time for real-time UX. |

**Next up — Module A.3: Prompt Engineering Fundamentals**

Now that you can call a model and control its output, the next module focuses on *what to say to it* — zero-shot, few-shot, chain-of-thought, and how to structure prompts for reliability.

---
*Model used: `HuggingFaceTB/SmolLM2-135M-Instruct`. All mechanics generalise directly to larger models and cloud APIs — only the `chat()` wrapper changes.*